**Depends on conda environment: fastq2matrix (see .env/fastq2matrix.yml)**

# 1

check the mitochondrial:

In [1]:
from pyfaidx import Fasta

Lzep_fa = Fasta("./ref/Lasioglossum_zephyrus.fasta")

In [2]:
for name in Lzep_fa.keys():
    print(name)

LZEP_unplaced_11829
LZEP_unplaced_7288
LZEP_unplaced_11828
LZEP_unplaced_8237
LZEP_unplaced_10352
LZEP_unplaced_8236
LZEP_unplaced_8235
LZEP_unplaced_8234
LZEP_unplaced_10163
LZEP_unplaced_8584
LZEP_unplaced_10353
LZEP_unplaced_9645
LZEP_unplaced_11823
LZEP_unplaced_11822
LZEP_unplaced_10354
LZEP_unplaced_9644
LZEP_unplaced_11801
LZEP_unplaced_14363
LZEP_unplaced_9560
LZEP_unplaced_10355
LZEP_unplaced_12980
LZEP_unplaced_9647
LZEP_unplaced_9230
LZEP_unplaced_9231
LZEP_unplaced_9232
LZEP_unplaced_9233
LZEP_unplaced_9234
LZEP_unplaced_9235
LZEP_unplaced_9236
LZEP_unplaced_9237
LZEP_unplaced_9238
LZEP_unplaced_9239
LZEP_unplaced_10356
LZEP_unplaced_9646
LZEP_unplaced_11778
LZEP_unplaced_9151
LZEP_unplaced_5551
LZEP_unplaced_5550
LZEP_unplaced_5553
LZEP_unplaced_5552
LZEP_unplaced_5555
LZEP_unplaced_5554
LZEP_unplaced_5557
LZEP_unplaced_5556
LZEP_unplaced_5559
LZEP_unplaced_5558
LZEP_unplaced_10600
LZEP_unplaced_10601
LZEP_unplaced_10606
LZEP_unplaced_10607
LZEP_unplaced_10604
LZEP_unplace

In [3]:
for name in Lzep_fa.keys():
    if name.startswith("LZEP_chr") or  name.startswith("LZEPmito"):
        print(f"{name}: {len(Lzep_fa[name])}")

LZEP_chr_14: 14608330
LZEP_chr_10: 23461232
LZEP_chr_11: 20746387
LZEP_chr_12: 7716444
LZEP_chr_13: 19623023
LZEP_chr_6: 12155835
LZEP_chr_7: 23416501
LZEP_chr_4: 14651701
LZEP_chr_5: 13343356
LZEP_chr_2: 25729413
LZEP_chr_3: 12391266
LZEP_chr_1: 14459315
LZEP_chr_8: 11150181
LZEP_chr_9: 20394881
LZEPmito: 14205


# 2

Examine the CDS and exon start site:

In [4]:
%%bash

gppy txinfo -g "./ref/Lasioglossum_zephyrus.gtf" > ./metadata/Lzep_txinfo.tsv

In [5]:
import pandas as pd

Lzep_txinfo = pd.read_csv("./metadata/Lzep_txinfo.tsv", sep="\t")
Lzep_txinfo

,tx_name,gene_id,chrom,strand,nexon,tx_len,cds_len,utr5_len,utr3_len,gene_biotype,gene_name
0,LZEP_07469-RA,LZEP_07469,LZEP_chr_1,+,3,651,381,78,192,protein_coding,LZEP_07469
1,LZEP_07470-RA,LZEP_07470,LZEP_chr_1,+,12,5651,2334,114,3203,protein_coding,LZEP_07470
2,LZEP_07471-RA,LZEP_07471,LZEP_chr_1,+,3,1885,1011,442,432,protein_coding,LZEP_07471
3,LZEP_07477-RA,LZEP_07477,LZEP_chr_1,+,4,3718,2589,651,478,protein_coding,LZEP_07477
4,LZEP_07478-RA,LZEP_07478,LZEP_chr_1,+,5,1796,456,1340,0,protein_coding,LZEP_07478
...,...,...,...,...,...,...,...,...,...,...,...
14041,nad3_t1,nad3,LZEPmito,+,1,354,0,0,0,protein_coding,MT-nad3
14042,nad4_t1,nad4,LZEPmito,-,1,1339,0,0,0,protein_coding,MT-nad4
14043,nad4l_t1,nad4l,LZEPmito,-,1,255,0,0,0,protein_coding,MT-nad4l
14044,nad5_t1,nad5,LZEPmito,-,1,1583,0,0,0,protein_coding,MT-nad5


In [6]:
utr5_missing = ((Lzep_txinfo["utr5_len"] == 0) & 
                (Lzep_txinfo["cds_len"] != 0)).sum() / \
               (Lzep_txinfo["cds_len"] != 0).sum()
print(utr5_missing)

0.6081379605216276


In [7]:
utr3_missing = ((Lzep_txinfo["utr3_len"] == 0) & 
                (Lzep_txinfo["cds_len"] != 0)).sum() / \
               (Lzep_txinfo["cds_len"] != 0).sum()
print(utr3_missing)

0.5893964227178793


In [8]:
both_missing = ((Lzep_txinfo["utr3_len"] == 0) & 
                (Lzep_txinfo["utr5_len"] == 0) &
                (Lzep_txinfo["cds_len"] != 0)).sum() / \
               (Lzep_txinfo["cds_len"] != 0).sum()
print(both_missing)

0.46369272429273856


In [9]:
either_missing = (((Lzep_txinfo["utr3_len"] == 0) | (Lzep_txinfo["utr5_len"] == 0)) & # 注意!! 必须将 | 括起来, 因为 & 的优先级比 | 大
                (Lzep_txinfo["cds_len"] != 0)).sum() / \
               (Lzep_txinfo["cds_len"] != 0).sum()
print(either_missing)

0.7338416589467683
